# Homework 2 — Problem 1:

In [1]:
# imports + setting random seed for reproducibility
import math, copy, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cpu"

## 1(a)

We have $g: R^p \rightarrow R^2$. Note $f_\mu(x)$ must be any real number and $f_\sigma(x)$ must be strictly positive:

$$f_\mu(x) = g_1(x), \qquad f_\sigma(x) = \exp(g_2(x)) + \varepsilon$$

In [4]:
# Creating MLP given the above link functions
class HeteroMLP(nn.Module):
    # g: R^p -> R^2, followed by the link functions from 1(a)
    def __init__(self, p, h=128, hetero=True, eps=1e-4):
        super().__init__()
        self.hetero, self.eps = hetero, eps
        self.body = nn.Sequential(nn.Linear(p, h), nn.ReLU(),
                                  nn.Linear(h, h), nn.ReLU())
        self.head = nn.Linear(h, 2 if hetero else 1)
        if not hetero:                       # one global (learned) sigma, same for every x
            self.raw_sigma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        out = self.head(self.body(x))
        mu = out[:, 0]                                        # identity link f'n
        if self.hetero:
            sigma = torch.exp(out[:, 1].clamp(-7, 7)) + self.eps        # exponential link f'n
        else:
            sigma = torch.exp(self.raw_sigma.clamp(-7, 7)).expand_as(mu) + self.eps
        return mu, sigma

## 1(b)

Under the model $Y | X=x \sim N(f_\mu(x), f_\sigma(x)^2)$, the log-density of one observation is $-log(\sigma) - 0.5*log(2\pi) - (y-\mu)^2 / (2\sigma^2)$. Maximizing the likelihood is the same as minimizing average negative log-likelihood:

$$\mathcal L(\theta)=\frac1n\sum_{i=1}^n\Big[\log f_\sigma(x_i)+\frac{(y_i-f_\mu(x_i))^2}{2 f_\sigma(x_i)^2}\Big]+\tfrac12\log 2\pi$$

In [7]:
# function for negative log-likelihood
def nll(mu, sigma, y):
    return (torch.log(sigma) + (y - mu) ** 2 / (2 * sigma ** 2)).mean() + 0.5 * math.log(2 * math.pi)

## 1(c)

* Features are already standardized (given). Target = `trip_duration` in minutes.
* **70/15/15 train/validation/test random split** with a fixed seed. Train fits the weights, validation drives early stopping, test is touched only for the final report.
* The target is standardized **using training statistics only** (no leakage), which gives the network a sensible scale (sigma starts near 0.69 ≈ 1). All reported metrics are converted back to minutes.

In [10]:
taxi_df = pd.read_parquet("/Users/rohansivakumar/Desktop/AI:ML/DATA/taxi_trips.parquet")
TARGET = "trip_duration"
FEATURES = [c for c in taxi_df.columns if c != TARGET]
print(taxi_df.shape, FEATURES)

(523554, 8) ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'min_of_day', 'day_of_week', 'day_of_year']


In [12]:
# 70% train
train_df, tmp_df = train_test_split(taxi_df, test_size=0.30, random_state=SEED)
# Splitting 30% test in half, 15% for validation, 15% for testing
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))
print(taxi_df[TARGET].describe())
print("fraction of trips < 45 min:", (taxi_df[TARGET] < 45).mean())

y_mean, y_std = train_df[TARGET].mean(), train_df[TARGET].std()

def to_tensors(d):
    X = torch.tensor(d[FEATURES].values, dtype=torch.float32, device=device)
    y = torch.tensor((d[TARGET].values - y_mean) / y_std, dtype=torch.float32, device=device)
    return X, y

Xtr, ytr = to_tensors(train_df); Xva, yva = to_tensors(val_df); Xte, yte = to_tensors(test_df)
y_test_raw = test_df[TARGET].values

366487 78533 78534
count    523554.000000
mean         10.413050
std           7.753304
min           0.033386
25%           4.749882
50%           8.631456
75%          13.928152
max         227.942612
Name: trip_duration, dtype: float64
fraction of trips < 45 min: 0.9956012178304434


### Baselines (Beatable models)

1. Constant: predict the training mean with the training std as sigma (the "know nothing" model).
2. Linear regression + constant sigma: captures any linear signal; sigma = std of training residuals.
3. Homoscedastic MLP (trained below): same network but a single global sigma. This is the *key* baseline: if the heteroscedastic MLP does not beat it on test NLL, the sigma head adds nothing.

Metrics: RMSE, MAE, mean **NLL** (the proper score for a distributional prediction), and the empirical coverage of the nominal 95% interval `mu ± 1.96 sigma` (should be ≈ 0.95 if sigma is well calibrated).

In [15]:
def summarize(name, y, mu, sigma):
    nll_raw = np.mean(0.5 * np.log(2 * np.pi) + np.log(sigma) + (y - mu) ** 2 / (2 * sigma ** 2))
    return dict(model=name, RMSE=np.sqrt(np.mean((y - mu) ** 2)), MAE=np.mean(np.abs(y - mu)),
                NLL=nll_raw, cover95=np.mean(np.abs(y - mu) <= 1.96 * sigma))

preds = {}   # name -> (mu, sigma) on the test set, in minutes
n_te = len(test_df)

# Baseline 1: constant mean / constant std
preds["const"] = (np.full(n_te, train_df[TARGET].mean()), np.full(n_te, train_df[TARGET].std()))

# Baseline 2: linear regression with constant sigma
lin = LinearRegression().fit(train_df[FEATURES].values, train_df[TARGET].values)
resid_sd = np.std(train_df[TARGET].values - lin.predict(train_df[FEATURES].values))
preds["linear+const sigma"] = (lin.predict(test_df[FEATURES].values), np.full(n_te, resid_sd))

In [17]:
def fit_mlp(hetero, h=128, epochs=40, bs=2048, lr=1e-3, patience=6, warmup=2, seed=SEED, verbose=True):
    torch.manual_seed(seed)
    model = HeteroMLP(Xtr.shape[1], h=h, hetero=hetero).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, best_state, bad, hist = float("inf"), None, 0, []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr), device=device)
        tr_loss = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            mu, sigma = model(Xtr[idx])
            loss = F.mse_loss(mu, ytr[idx]) if ep < warmup else nll(mu, sigma, ytr[idx])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tr_loss += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            val = nll(*model(Xva), yva).item()
        hist.append((tr_loss / len(Xtr), val))
        if verbose: print(f"epoch {ep+1:2d}  train loss {hist[-1][0]:.4f}  val NLL {val:.4f}")
        if ep >= warmup:
            if val < best_val - 1e-4:
                best_val, best_state, bad = val, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= patience: break
    model.load_state_dict(best_state)
    return model, hist

@torch.no_grad()
def predict(model, X):
    model.eval()
    mu, sigma = model(X)
    return mu.cpu().numpy() * y_std + y_mean, sigma.cpu().numpy() * y_std   # back to minutes

print("--- homoscedastic MLP (constant sigma) ---")
model_homo, hist_homo = fit_mlp(hetero=False)
print("--- heteroscedastic MLP ---")
model_het, hist_het = fit_mlp(hetero=True)

--- homoscedastic MLP (constant sigma) ---
epoch  1  train loss 0.4722  val NLL 1.0879
epoch  2  train loss 0.3203  val NLL 1.0674
epoch  3  train loss 1.0080  val NLL 0.9472
epoch  4  train loss 0.9048  val NLL 0.8578
epoch  5  train loss 0.8307  val NLL 0.7967
epoch  6  train loss 0.7853  val NLL 0.7630
epoch  7  train loss 0.7605  val NLL 0.7503
epoch  8  train loss 0.7477  val NLL 0.7407
epoch  9  train loss 0.7408  val NLL 0.7320
epoch 10  train loss 0.7354  val NLL 0.7350
epoch 11  train loss 0.7304  val NLL 0.7245
epoch 12  train loss 0.7286  val NLL 0.7250
epoch 13  train loss 0.7244  val NLL 0.7213
epoch 14  train loss 0.7215  val NLL 0.7191
epoch 15  train loss 0.7193  val NLL 0.7199
epoch 16  train loss 0.7169  val NLL 0.7202
epoch 17  train loss 0.7167  val NLL 0.7199
epoch 18  train loss 0.7146  val NLL 0.7146
epoch 19  train loss 0.7123  val NLL 0.7137
epoch 20  train loss 0.7106  val NLL 0.7189
epoch 21  train loss 0.7083  val NLL 0.7099
epoch 22  train loss 0.7079  val 

In [19]:
preds["MLP (const sigma)"] = predict(model_homo, Xte)
preds["MLP (hetero sigma)"] = predict(model_het, Xte)

results = pd.DataFrame([summarize(k, y_test_raw, *v) for k, v in preds.items()]).set_index("model")
display(results.round(3))

,RMSE,MAE,NLL,cover95
model,,,,
const,7.688,5.664,3.459,0.960
linear+const sigma,7.473,5.554,3.430,0.958
MLP (const sigma),3.743,2.382,2.739,0.957
MLP (hetero sigma),3.900,2.471,2.628,0.961


We see the RMSE, MAE, and NLL values decrease as the model 'complexity' approaches the heteroskedastic MLP model.

## 1(d)

Given the fitted model, $Y | x \sim N(f_\mu(x), f_\sigma(x)^2)$, so

$$P(Y < 45 \mid x) = \Phi\!\left(\frac{45 - f_\mu(x)}{f_\sigma(x)}\right)$$

In [29]:
# implementation of above formula
def prob_under(mu, sigma, threshold=45.0):
    return norm.cdf((threshold - mu) / sigma)

## 1(e)

In [32]:
label = (y_test_raw < 45).astype(float)
base_rate = (train_df[TARGET] < 45).mean()

# includes brier score, log-loss, and accuracy
def prob_metrics(name, p):
    p_c = np.clip(p, 1e-12, 1 - 1e-12)
    return dict(model=name,
                Brier=np.mean((p - label) ** 2),
                log_loss=-np.mean(label * np.log(p_c) + (1 - label) * np.log(1 - p_c)),
                accuracy=np.mean((p > 0.5) == (label == 1)))

probs = {"base rate (constant p)": np.full(n_te, base_rate)}
for k in ["const", "linear+const sigma", "MLP (const sigma)", "MLP (hetero sigma)"]:
    probs[k] = prob_under(*preds[k])

prob_table = pd.DataFrame([prob_metrics(k, p) for k, p in probs.items()]).set_index("model")
print("majority-class accuracy:", max(label.mean(), 1 - label.mean()))
display(prob_table.round(5))

majority-class accuracy: 0.9956833982733593


,Brier,log_loss,accuracy
model,,,
base rate (constant p),0.00430,0.02781,0.99568
const,0.00432,0.05322,0.99568
linear+const sigma,0.00432,0.04452,0.99568
MLP (const sigma),0.00301,0.03265,0.99666
MLP (hetero sigma),0.00307,0.02164,0.99628


Note that the accuracy is ≈ 99.6% for every model, so accuracy alone cannot distinguish model efficacy. Thus, I'd use the Brier/log-loss to make a decision. Note that the Brier score and log-loss are lower (better) for the MLPs compared to the other baseline models. However, there is negligible difference in Brier score between the homo and heteroskedastic MLPs. That said, there is an improvement in log-loss once accounting for heteroskedasticity.